# Option 4 - Smart Home Remote

The smart home remote should be able to execute actions such as manipulating the state of light switches, thermostats and door locks. 

Using Python's library for abstract classes and methods where it is commonly used to create interfaces. "abc" refers to a base class when inherited from creates an abstract class.

In [8]:
from abc import ABC, abstractmethod

Create the command interface with the execute and undo methods. 

In [9]:
# Command Interface
class Command(ABC):
    @abstractmethod
    def execute(self):
        """Execute the command."""
        pass

    def undo(self):
        """Undo the command."""
        pass

Receivers contain the logic for controlling the smart devices themselves, an example being turning a specific light on or off.

In [10]:
# Receivers
class Light:
    def __init__(self, location):
        # Location of the light in the house. Can be 'Living Room', 'Kitchen', etc. and can be used to group lights together.
        self.location = location

        # Initial state of the light.
        self.is_on = False

    def turn_on(self):
        """Turn on the light and print a message confirming the light has been turned on."""
        self.is_on = True
        print(f"{self.location} light turned on")

    def turn_off(self):
        """Turn off the light and print a message confirming the light has been turned off."""
        self.is_on = False
        print(f"{self.location} light turned off")

class Thermostat:
    def __init__(self):
        # Initial temperature of the thermostat.
        self.temperature = 21

    def set_temperature(self, temperature):
        """Set the temperature of the thermostat and print a message confirming the new temperature."""
        self.temperature = temperature
        print(f"Thermostat temperature set to {self.temperature}°C")

class DoorLock:
    def __init__(self, location):
        # Location of the door lock in the house. Can be 'Front Door', 'Back Door', etc. and can be used to group locks together (if applicable).
        self.location = location
        # Initial state of the door lock.
        self.locked = False

    def lock(self):
        """Lock the door and print a message confirming the door has been locked."""
        self.locked = True
        print(f"{self.location} door locked")

    def unlock(self):
        """Unlock the door and print a message confirming the door has been unlocked."""
        self.locked = False
        print(f"{self.location} door unlocked")

Concrete Command classes are used to wrap calls to the respective receiver that contains the logic. An example is the concrete command class using the thermostat's logic to set the temperature to 25°c.

In [11]:
# Concrete Commands
class LightOnCommand(Command):
    def __init__(self, light):
        # Initialise the LightOnCommand with a specific light receiver.
        self.light = light

    def execute(self):
        """Execute the command to turn on the light."""
        self.light.turn_on()

    def undo(self):
        """Undo the command to turn off the light."""
        self.light.turn_off()

class LightOffCommand(Command):
    def __init__(self, light):
        # Initialise the LightOffCommand with a specific light receiver.
        self.light = light

    def execute(self):
        """Execute the command to turn off the light."""
        self.light.turn_off()

    def undo(self):
        """Undo the command to turn on the light."""
        self.light.turn_on()


class SetThermostatCommand(Command):
    def __init__(self, thermostat, new_temperature):
        # Initialise the SetThermostatCommand with a specific thermostat receiver and the new temperature to set.
        self.thermostat = thermostat
        self.new_temperature = new_temperature
        self.previous_temperature = thermostat.temperature

    def execute(self):
        """Execute the command to set the thermostat to the new temperature, saving the previous temperature for undo functionality."""
        self.previous_temperature = self.thermostat.temperature
        self.thermostat.set_temperature(self.new_temperature)

    def undo(self):
        """Undo the command to set the thermostat to the previous temperature."""
        self.thermostat.set_temperature(self.previous_temperature)

class LockDoorCommand(Command):
    def __init__(self, door_lock):
        # initialise the LockDoorCommand with a specific door lock receiver.
        self.door_lock = door_lock

    def execute(self):
        """Execute the command to lock the door."""
        self.door_lock.lock()

    def undo(self):
        """Undo the command to unlock the door."""
        self.door_lock.unlock()


class UnlockDoorCommand(Command):
    def __init__(self, door_lock):
        # initialise the UnlockDoorCommand with a specific door lock receiver.
        self.door_lock = door_lock

    def execute(self):
        """Execute the command to unlock the door."""
        self.door_lock.unlock()

    def undo(self):
        """Undo the command to lock the door."""
        self.door_lock.lock()

The RemoteControl acts as the command invoker. An interface for the client to press buttons that call the commands.

In [12]:
# Command Invoker
class RemoteControl:
    def __init__(self):
        # Initialise the RemoteControl with an empty dictionary of slots and an null last command.
        self.slots = {}
        self.last_command = None

    def set_command(self, slot, command):
        """Set the command for a specific slot in the remote control."""
        self.slots[slot] = command

    def press_button(self, slot):
        """Method to simulate pressing a button on the remote control."""
        # Get the command associated with the given slot and execute it, while also saving it as the last command for potential undo functionality.
        command = self.slots.get(slot)
        print(f"\nButton {slot} pressed")

        # Execute the command and save it as the last command for potential undo functionality.
        command.execute()
        self.last_command = command

    def press_undo(self):
        print("\nUndo pressed")
        # If the last command exists, call its undo method to reverse the last action.
        if self.last_command:
            self.last_command.undo()


# Test Runs

- Create the receivers for each device and the location for the light and door.
- Create the RemoteControl interface for the commands to be executed from.
- Set commands to the RemoteControl interface that execute when pressed.

In [16]:
living_room_light = Light("Living room")
thermostat = Thermostat()
front_door = DoorLock("Front")

remote = RemoteControl()

remote.set_command(1, LightOnCommand(living_room_light))
remote.set_command(2, LightOffCommand(living_room_light))
remote.set_command(3, SetThermostatCommand(thermostat, 22))
remote.set_command(4, LockDoorCommand(front_door))
remote.set_command(5, UnlockDoorCommand(front_door))

In [ ]:
remote.press_button(1)
remote.press_button(3)
remote.press_button(4)
remote.press_undo()
remote.press_button(2)